In [1]:
# install or upgrade required packages
%pip install mlflow==3.3.1 langchain langchain-community langchain-openai python-dotenv sentence-transformers faiss-cpu --upgrade

  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached faiss_cpu-1.12.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.1 kB)
  Using cached numpy-2.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached langchain_community-0.3.29-py3-none-any.whl (2.5 MB)
Using cached faiss_cpu-1.12.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (31.4 MB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/conda/lib/python3.11/site-packages/numpy/__init__.cython-30.pxd'
Consider using the `--user` option or check the permissions.

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Imports & setup
import os
import mlflow
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
import mlflow.genai

# Load environment variables from .env file
# Ensure you have your Azure credentials in a .env file in the project root
load_dotenv('../.env')

# --- Configuration ---
MLFLOW_TRACKING_URI = "http://mlflow:5000"
EXPERIMENT_NAME = "Building-Companion-Prompt-Engineering"
VECTOR_STORE_PATH = "../artifacts/vector_store"

# --- MLflow Setup ---
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ MLflow and libraries are ready.")
print(f"Tracking experiments in: '{EXPERIMENT_NAME}'")

✅ MLflow and libraries are ready.
Tracking experiments in: 'Building-Companion-Prompt-Engineering'


In [3]:
# load the vector store
print("Loading embedding model...")
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)

print(f"Loading vector store from: {VECTOR_STORE_PATH}")
db = FAISS.load_local(VECTOR_STORE_PATH, embedder, allow_dangerous_deserialization=True)
retriever = db.as_retriever()

print("✅ Knowledge base is loaded and ready.")

Loading embedding model...


/tmp/ipykernel_1973/1439560388.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedder = HuggingFaceEmbeddings(model_name=model_name)


Loading vector store from: ../artifacts/vector_store
✅ Knowledge base is loaded and ready.


In [4]:
# Define the prompt templates
prompt_templates = {
    "legal_expert": """
    You are an AI assistant specialized in Portuguese building regulations based on the provided document.
    Use the following pieces of context to answer the question at the end.
    If you don't know the answer from the context, just say that you don't know, don't try to make up an answer.
    Provide a concise and direct answer based strictly on the provided text. Cite the article number if possible.

    Context: {context}

    Question: {question}

    Answer (in Portuguese):
    """,
    "homeowner_guide": """
    You are a helpful AI assistant for homeowners in Portugal.
    Use the information from the provided legal document to answer the user's question in a simple, easy-to-understand way.
    Explain the key points without using complex legal jargon.

    Based on the regulations: {context}

    Here is the answer to your question: {question}

    Helpful Answer (in Portuguese):
    """
}

print(f"Defined {len(prompt_templates)} prompt templates.")

Defined 2 prompt templates.


In [5]:
# Cell 4.5: Register Prompts in MLflow Prompt Registry

def register_prompt_in_mlflow(name, template, tags):
    """Registers a single prompt in MLflow and returns its URI."""
    try:
        # The register_prompt function creates a new version if the prompt name already exists.
        registered_prompt = mlflow.register_prompt(
            name=name,
            template=template,
            tags=tags
        )
        # Use the actual version number returned by MLflow
        uri = f"prompts:/{name}/{registered_prompt.version}"
        print(f"✅ Successfully registered '{name}'. URI: {uri}")
        return uri
    except Exception as e:
        print(f"⚠️  Failed to register '{name}'. Error: {e}")
        return None

# --- Register all our defined prompts ---
print("📝 Registering all prompt templates...")
registered_prompt_uris = {}
for name, text in prompt_templates.items():
    tags = {"use_case": "legal_chatbot", "persona": name}
    uri = register_prompt_in_mlflow(name, text, tags)
    if uri:
        registered_prompt_uris[name] = uri

print("\n🎉 Prompt registration process complete.")

📝 Registering all prompt templates...


/tmp/ipykernel_1973/4157525007.py:7: FutureWarning: The `mlflow.register_prompt` API is moved to the `mlflow.genai` namespace. Please use `mlflow.genai.register_prompt` instead. The original API will be removed in the future release.
  registered_prompt = mlflow.register_prompt(
2025/09/21 11:08:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: legal_expert, version 1


✅ Successfully registered 'legal_expert'. URI: prompts:/legal_expert/1


2025/09/21 11:08:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: homeowner_guide, version 1


✅ Successfully registered 'homeowner_guide'. URI: prompts:/homeowner_guide/1

🎉 Prompt registration process complete.


In [6]:
# Cell 5 (Updated): Run Experiments Using Prompts from the Registry

def run_chatbot_experiment_from_registry(prompt_uri, question):
    """Runs a query using a prompt loaded from the MLflow Registry."""
    
    # 1. Load the versioned prompt object from MLflow
    mlflow_prompt_object = mlflow.genai.load_prompt(prompt_uri)

    # 2. Extract the template string and create a LangChain PromptTemplate object
    from langchain.prompts import PromptTemplate
    prompt_from_registry = PromptTemplate(
        template=mlflow_prompt_object.template,
        input_variables=["context", "question"]
    )

    llm = AzureChatOpenAI(
        deployment_name=os.getenv("AZURE_DEPLOYMENT_NAME"),
        openai_api_version="2024-02-15-preview"
    )

    # 3. Use the new, correct LangChain prompt object in the chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_from_registry}
    )

    return qa_chain.invoke({"query": question})

In [7]:
# --- Let's run our experiments! ---
test_questions = [
    "Posso construir uma cave para habitação?",
    "Qual é a altura mínima para o pé-direito de uma loja comercial?",
    "Qual a largura mínima de uma escada num prédio com 4 apartamentos?"
]

for prompt_name, uri in registered_prompt_uris.items():
    with mlflow.start_run(run_name=f"registry_prompt_test_{prompt_name}"):
        print(f"\n--- Testing Prompt from Registry: {prompt_name} ---")
        mlflow.log_param("prompt_name", prompt_name)
        mlflow.log_param("prompt_uri", uri) # Log the exact prompt version used

        for i, question in enumerate(test_questions):
            mlflow.log_param(f"question_{i}", question)

            result = run_chatbot_experiment_from_registry(uri, question)
            answer = result['result']

            mlflow.log_text(answer, f"answer_{i}.txt")
            print(f"Q: {question}")
            print(f"A: {answer}\n")

print("✅ All experiments completed. Check the MLflow UI at http://localhost:5001")


--- Testing Prompt from Registry: legal_expert ---
Q: Posso construir uma cave para habitação?
A: De acordo com o regulamento apresentado, é permitido construir caves para habitação desde que sejam satisfeitas todas as condições de salubridade previstas para os andares de habitação. Além disso, devem ser observados os seguintes requisitos específicos (Artigo 79.º):

a) A cave deve ter, pelo menos, uma parede exterior completamente desafogada a partir de 0,15 m abaixo do pavimento interior;  
b) Compartimentos habitáveis devem ser contíguos à fachada completamente desafogada;  
c) Devem ser adotadas medidas construtivas para garantir proteção contra infiltrações, humidade telúrica e emanações subterrâneas;  
d) O escoamento de esgotos deverá ser feito por gravidade.  

Se essas condições forem cumpridas, a cave poderá ser usada para habitação.

Q: Qual é a altura mínima para o pé-direito de uma loja comercial?
A: A altura mínima para o pé-direito de uma loja comercial é de 2,70 metros,